In [1]:
import pandas as pd

df = pd.read_csv("../../data/cleaned_data/merged_2021_2025.csv")

# Parse datetime columns
datetime_cols = ["Opened", "Closed Date 1", "Closed Date 2", "closed"]
for col in datetime_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Ensure numeric types
df["resolution_time_hours"] = pd.to_numeric(df["resolution_time_hours"], errors="coerce")
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lng"] = pd.to_numeric(df["lng"], errors="coerce")

# Standardize boolean columns
bool_cols = ["has_closure_time", "ok"]
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().map({"true": True, "false": False})


C:\Users\shrey\AppData\Local\Temp\ipykernel_35308\2146284045.py:3: DtypeWarning: Columns (1,3,4,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../data/cleaned_data/merged_2021_2025.csv")


In [2]:
print("Length of df before cleanup:", len(df))

# Fill missing or empty Description using Short Description
df["Description"] = df["Description"].fillna("").str.strip()
df["Short Description"] = df["Short Description"].fillna("").str.strip()

df.loc[df["Description"] == "", "Description"] = df.loc[df["Description"] == "", "Short Description"]

# Remove rows where Description is still empty
df = df[df["Description"].notna()]
df = df[df["Description"].str.strip() != ""]

# Drop duplicates
df = df.drop_duplicates(subset=["Description", "Address", "Opened"])

print("Length of df after cleanup:", len(df))

Length of df before cleanup: 443319
Length of df after cleanup: 442395


In [3]:
#Fetch unique data

df["Description_norm"] = (
    df["Description"]
    .str.strip()
    .str.lower()
)

unique_desc_df = df[["Description_norm"]].drop_duplicates().reset_index(drop=True)

print("Original rows:", len(df))
print("Unique descriptions:", len(unique_desc_df))

Original rows: 442395
Unique descriptions: 1060


In [4]:
import re

def normalize(text):
    text = text.lower().strip()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

unique_desc_df["clean_text"] = unique_desc_df["Description_norm"].apply(normalize)


**Clustering techniques to categorize different concerns / requests raise**

In [6]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer

c:\Users\shrey\anaconda3\envs\geo\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Method 1:** TF-IDF

In [7]:
tfidf = TfidfVectorizer(
    max_features=2000,
    ngram_range=(1,2),
    stop_words="english"
)

X_tfidf = tfidf.fit_transform(unique_desc_df["clean_text"])

k = 10
kmeans_tfidf = KMeans(n_clusters=k, random_state=42)
unique_desc_df["cluster_tfidf"] = kmeans_tfidf.fit_predict(X_tfidf)

sil_tfidf = silhouette_score(X_tfidf, unique_desc_df["cluster_tfidf"])
print("TF-IDF Silhouette:", sil_tfidf)

TF-IDF Silhouette: 0.025117845720966026


**Method 2:** Sentence Embedding + K-means

In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
X_embed = model.encode(unique_desc_df["clean_text"].tolist(), show_progress_bar=True)
kmeans_embed = KMeans(n_clusters=k, random_state=42)
unique_desc_df["cluster_embed"] = kmeans_embed.fit_predict(X_embed)

sil_embed = silhouette_score(X_embed, unique_desc_df["cluster_embed"])
print("Embedding Silhouette:", sil_embed)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 579.01it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 34/34 [00:02<00:00, 15.88it/s]
c:\Users\shrey\anaconda3\envs\geo\lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=5.
  warnings.warn(


Embedding Silhouette: 0.06685686111450195


**Method 3:** Sentence Embedding + Hierarchical CLustering

In [9]:
from sklearn.cluster import AgglomerativeClustering

agg = AgglomerativeClustering(n_clusters=k)
unique_desc_df["cluster_hier"] = agg.fit_predict(X_embed)

sil_hier = silhouette_score(X_embed, unique_desc_df["cluster_hier"])
print("Hierarchical Silhouette:", sil_hier)


Hierarchical Silhouette: 0.038824599236249924


**Results based on the three technqiues mentioned:**

TF-IDF Silhouette: 0.025117845720966026

Embedding Silhouette: 0.06685686111450195

Hierarchical Silhouette: 0.038824599236249924

Higher silhouette value means better grouping. Thus, the groupings are not good.

**New Approach:** Semantic grouping + Domain and Severity separation

In [10]:
categories = [
    "Waste Management",
    "Water & Sewer",
    "Road & Infrastructure",
    "Code Enforcement",
    "Licensing & Permits",
    "Administrative / Account Services",
    "Public Safety",
    "Other"
]

category_anchors = {
    "Waste Management": "garbage recycling bulk trash cart yard trimmings waste pickup",
    "Water & Sewer": "water sewer leak meter no water account bill adjustment",
    "Road & Infrastructure": "pothole street repair sidewalk road damage",
    "Code Enforcement": "junk debris violation private property complaint",
    "Licensing & Permits": "business license permit renewal application",
    "Administrative / Account Services": "account information transfer close billing inquiry",
    "Public Safety": "hazard dangerous emergency complaint",
}

In [11]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

# Embed unique descriptions
desc_embeddings = model.encode(
    unique_desc_df["Description_norm"].tolist(),
    show_progress_bar=True
)

# Embed anchor texts
anchor_embeddings = model.encode(list(category_anchors.values()))


similarity_matrix = cosine_similarity(desc_embeddings, anchor_embeddings)

# Get best matching category index
best_match_idx = np.argmax(similarity_matrix, axis=1)

# Map index back to category name
category_names = list(category_anchors.keys())
unique_desc_df["domain"] = [category_names[i] for i in best_match_idx]



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 519.36it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 34/34 [00:03<00:00,  8.93it/s]


In [13]:
def issue_type(text):
    text = text.lower()

    if any(w in text for w in ["leak", "backup", "burst", "hazard", "flood", "sinkhole"]):
        return "Hazard / Emergency"

    elif any(w in text for w in ["pothole", "broken", "damaged", "repair"]):
        return "Infrastructure Damage"

    elif any(w in text for w in ["missed", "missing", "replacement", "pickup"]):
        return "Service Failure"

    elif any(w in text for w in ["request", "install", "new", "schedule"]):
        return "Service Request"

    elif any(w in text for w in ["bill", "account", "payment", "balance", "refund"]):
        return "Billing / Account"

    else:
        return "Information / Other"
    
unique_desc_df["issue_type"] = unique_desc_df["Description_norm"].apply(issue_type)

In [14]:
unique_desc_df.to_csv("../../data/cleaned_data/categories_domain_and_issue.csv", index=False)


In [15]:

unique_desc_df['domain'].value_counts()


domain
Public Safety                        218
Code Enforcement                     197
Licensing & Permits                  160
Waste Management                     159
Water & Sewer                        154
Road & Infrastructure                107
Administrative / Account Services     65
Name: count, dtype: int64

In [16]:
unique_desc_df['issue_type'].value_counts()


issue_type
Information / Other      720
Service Request          149
Infrastructure Damage     60
Billing / Account         53
Service Failure           48
Hazard / Emergency        30
Name: count, dtype: int64

In [17]:
# Map back to full dataset

desc_to_domain = dict(
    zip(unique_desc_df["Description_norm"], unique_desc_df["domain"])
)

df["domain"] = df["Description_norm"].map(desc_to_domain)


desc_to_issue_type = dict(
    zip(unique_desc_df["Description_norm"], unique_desc_df["issue_type"])
)

df["issue_type"] = df["Description_norm"].map(desc_to_issue_type)


In [20]:
df.columns

Index(['Opened', 'Description', 'Address', 'Zip Code', 'Closed Date 1',
       'Closed Date 2', 'Status', 'Number', 'closed', 'resolution_time_hours',
       'has_closure_time', 'zip_clean', 'zip_int', 'address_norm', 'lat',
       'lng', 'ok', 'year', 'Short Description', 'Description_norm', 'domain',
       'issue_type'],
      dtype='object')

In [21]:
df["domain"].value_counts()


domain
Waste Management                     200616
Code Enforcement                      70178
Water & Sewer                         62066
Road & Infrastructure                 29705
Public Safety                         22827
Administrative / Account Services     13878
Licensing & Permits                   13861
Name: count, dtype: int64

In [22]:
df["issue_type"].value_counts()


issue_type
Information / Other      114549
Service Request          108078
Service Failure           89399
Infrastructure Damage     60636
Billing / Account         20302
Hazard / Emergency        20167
Name: count, dtype: int64

**Severity Score (Domain + Issue Type)**

In [18]:
domain_weight = {
    "Water & Sewer": 5,
    "Public Safety": 5,
    "Road & Infrastructure": 4,
    "Waste Management": 3,
    "Code Enforcement": 3,
    "Administrative / Account Services": 2,
    "Licensing & Permits": 1,
    "Other": 1
}

In [19]:
issue_weight = {
    "Hazard / Emergency": 5,
    "Infrastructure Damage": 4,
    "Service Failure": 3,
    "Service Request": 2,
    "Billing / Account": 1,
    "Information / Other": 1
}

In [20]:
df["domain_weight"] = df["domain"].map(domain_weight)
df["issue_weight"] = df["issue_type"].map(issue_weight)

df["severity_score"] = df["domain_weight"] * df["issue_weight"]

In [22]:
df.to_csv("../../data/cleaned_data/merged_data_with_domain_issue_type.csv", index=False)

In [31]:
baseline = (
    df.groupby(["domain", "issue_type"])["resolution_time_hours"]
    .mean()
    .reset_index()
    .rename(columns={"resolution_time_hours": "expected_resolution"})
)

In [32]:
df = df.merge(baseline, on=["domain", "issue_type"], how="left")

In [33]:
df["resolution_factor"] = (
    df["resolution_time_hours"] / df["expected_resolution"]
)

def resolution_score(factor):

    if factor <= 0.5:
        return 5   # extremely fast

    elif factor <= 0.8:
        return 4   # faster than average

    elif factor <= 1.2:
        return 3   # normal

    elif factor <= 1.8:
        return 2   # slow

    else:
        return 1   # very slow
    

df["resolution_score"] = df["resolution_factor"].apply(resolution_score)

In [34]:
df.to_csv("../data/geocoded_data/data_severity_resolution_score_assigned.csv", index=False)

In [35]:
df.columns

Index(['Opened', 'Description', 'Address', 'Zip Code', 'Closed Date 1',
       'Closed Date 2', 'Status', 'Number', 'closed', 'resolution_time_hours',
       'has_closure_time', 'zip_clean', 'zip_int', 'address_norm', 'lat',
       'lng', 'ok', 'year', 'Short Description', 'Description_norm', 'domain',
       'issue_type', 'domain_weight', 'issue_weight', 'severity_score',
       'expected_resolution', 'resolution_factor', 'resolution_score'],
      dtype='object')

In [ ]:
# category_scores = {

#     "Public Safety": 10,                       # emergency, danger
#     "Water & Sewer": 9,                        # flooding, sanitation risk
#     "Road & Infrastructure": 8,                # traffic, physical risk
#     "Waste Management": 7,                     # hygiene impact
#     "Code Enforcement": 6,                     # compliance issues
#     "Licensing & Permits": 5,                  # business impact
#     "Administrative / Account Services": 4,    # billing/info
#     "Other": 3                                 # uncategorized
    
# }

In [50]:
df["category_score"] = df["category"].map(category_scores)

In [51]:
print(df["category_score"].isna().sum())

0


In [52]:
max_score = max(category_scores.values())

df["category_score_norm"] = df["category_score"] / max_score

In [53]:
df.to_csv("../data/geocoded_data/merged_data_with_categories_score.csv", index=False)

In [54]:
df.columns

Index(['Opened', 'Description', 'Address', 'Zip Code', 'Closed Date 1',
       'Closed Date 2', 'Status', 'Number', 'closed', 'resolution_time_hours',
       'has_closure_time', 'zip_clean', 'zip_int', 'address_norm', 'lat',
       'lng', 'ok', 'year', 'Short Description', 'Description_norm',
       'category', 'category_score', 'category_score_norm'],
      dtype='object')